# 17 — BCG mouse → predicted human (no quantitative eval)

Push the BCG mouse cells (preprocessed in notebook 16) through the full-atlas-trained models (notebook 15 + sbatches), one model + flavor at a time, and write predicted human cells to disk. NO evaluation against real human BCG — that data is not yet integrated; this notebook produces predictions only.

**Pipeline per (flavor, model):**
1. Load trained `model.pt` from `cellot/cellot_gpu/results/atlas_full_{flavor}/{model}/cache/model.pt`.
2. Load `bcg_mouse_aligned_{flavor}_v07.h5ad` from `cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/`.
3. Forward pass:
   - **scGen**: encode → +shift (atlas-derived `code_means["human"] − code_means["mouse"]`) → decode → 1000-d gene-space prediction.
   - **IMPACT_CellOT**: encode → OT map `g(z)` → decode → 1000-d gene-space prediction.
4. Write `bcg_predicted_human_via_{model}_{flavor}.h5ad`.

**Qualitative UMAP**: per-flavor, recompute UMAP on the joint stack (atlas mouse, atlas human, BCG mouse, scGen-pred, IMPACT-pred) and color by source. No metrics — just "do predictions land in the human cloud?"

**Dependencies**: notebooks 15 + 16 done; `atlas_full_*` trainings finished.

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

BASE = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
RESULTS = BASE / "cellot/cellot_gpu/results"
DATA_DIR = BASE / "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg"
OUT_DIR = BASE / "speciesOT/baseline/analysis/bcg_prediction_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FLAVORS = ["seurat_v3", "pearson_residuals"]
MODELS = ["scgen", "impact_cellot"]

print("Working from", BASE)
print("flavors:", FLAVORS, "  models:", MODELS)

Working from /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT
flavors: ['seurat_v3', 'pearson_residuals']   models: ['scgen', 'impact_cellot']


## 1. Use the CellOT env to load models and run forward passes

The cellot library lives in the `CellOT` conda env (anndata 0.7, torch). We shell out to it via subprocess so the model loading uses the env it was trained in.

In [2]:
import subprocess

CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"

predict_script = r"""
# Run inside CellOT env. Loads a trained model and produces predictions on a custom AnnData.
import sys, os
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu")
import torch, numpy as np, pandas as pd, anndata as ad
from cellot.utils import load_config
from cellot.utils.loaders import load_model, load_data, resolve_device
from cellot.transport import transport_scgen, transport_cellot
from cellot.models.ae import compute_scgen_shift

flavor, model_name, atlas_data_path, bcg_data_path, results_dir, out_path = sys.argv[1:7]
 
# Load config + model
config_path = os.path.join(results_dir, "config.yaml")
config = load_config(config_path)
device = resolve_device(config)

# Patch ae_emb path (relative -> absolute) for IMPACT
if "ae_emb" in config.data:
    config.data.ae_emb.path = os.path.join(os.path.dirname(results_dir), "model-scgen")
# Patch data.path for the source the model was trained on (so compute_scgen_shift can re-walk training cells)
config.data.path = atlas_data_path

# For scGen and IMPACT, the load_model path is the same dispatch
model_kwargs = {}
if model_name == "scgen":
    from cellot.models import load_autoencoder_model
    a = ad.read(atlas_data_path)
    model_kwargs["input_dim"] = a.n_vars
    model, _ = load_autoencoder_model(config, restore=os.path.join(results_dir, "cache", "model.pt"),
                                      device=device, **model_kwargs)
    model.eval()
    # Compute / restore code_means from atlas training data
    if not hasattr(model, "code_means"):
        loader = load_data(config, return_as="loader")
        labels = loader.train.dataset.adata.obs[config.data.condition]
        compute_scgen_shift(model, loader.train.dataset, labels=labels, device=device)
elif model_name == "impact_cellot":
    # IMPACT_CellOT operates in the AE's latent space, so input_dim = latent_dim (50)
    latent_dim = config.model.get("latent_dim", 50)
    model, *_ = load_model(config, restore=os.path.join(results_dir, "cache", "model.pt"),
                           device=device, input_dim=latent_dim)
else:
    raise ValueError(model_name)

# Load BCG mouse cells
bcg = ad.read(bcg_data_path)
bcg_X = bcg.X
if hasattr(bcg_X, "toarray"):
    bcg_X = bcg_X.toarray()
inputs = torch.tensor(bcg_X, dtype=torch.float32).to(device)

# Forward pass
if model_name == "scgen":
    model.eval()
    shift = model.code_means["human"] - model.code_means["mouse"]
    codes = model.encode(inputs)
    pred = model.decode(codes + shift).detach().cpu().numpy()
else:
    # IMPACT_CellOT: encode via the AE first (config has ae_emb), then OT, then decode
    from cellot.models import load_autoencoder_model
    ae_config = load_config(os.path.join(config.data.ae_emb.path, "config.yaml"))
    ae_model, _ = load_autoencoder_model(ae_config, restore=os.path.join(config.data.ae_emb.path, "cache", "model.pt"),
                                         device=device, input_dim=bcg.n_vars)
    ae_model.eval()
    f, g = model
    g.eval()
    codes = ae_model.encode(inputs)
    transported = g.transport(codes.requires_grad_(True))
    pred = ae_model.decode(transported.detach()).detach().cpu().numpy()

# Wrap in AnnData and write
out = ad.AnnData(X=pred.astype(np.float32),
                 obs=bcg.obs.copy(),
                 var=bcg.var.copy())
out.write(out_path)
print(f"wrote {out_path}: shape={out.shape}, X mean={float(out.X.mean()):.4f}")
"""

# Run for each (flavor, model) combination
prediction_paths = {}
for flavor in FLAVORS:
    atlas_path = str(DATA_DIR / f"hvg_{flavor}_atlas_full_v07.h5ad")
    bcg_path = str(DATA_DIR / f"bcg_mouse_aligned_{flavor}_v07.h5ad")
    for model_name in MODELS:
        results_dir = str(RESULTS / f"atlas_full_{flavor}" / model_name)
        out_path = str(DATA_DIR / f"bcg_predicted_human_via_{model_name}_{flavor}.h5ad")
        print(f"\n=== {flavor} / {model_name} ===")
        try:
            subprocess.run(
                [CELLOT_PY, "-c", predict_script, flavor, model_name, atlas_path, bcg_path, results_dir, out_path],
                check=True
            )
            prediction_paths[(flavor, model_name)] = out_path
        except subprocess.CalledProcessError as e:
            print(f"  FAILED: {e}")

print(f"\nWrote {len(prediction_paths)} prediction files")


=== seurat_v3 / scgen ===
wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_predicted_human_via_scgen_seurat_v3.h5ad: shape=(1406, 1000), X mean=0.1155

=== seurat_v3 / impact_cellot ===
wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_predicted_human_via_impact_cellot_seurat_v3.h5ad: shape=(1406, 1000), X mean=0.1230

=== pearson_residuals / scgen ===
wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_predicted_human_via_scgen_pearson_residuals.h5ad: shape=(1406, 1000), X mean=0.4321

=== pearson_residuals / impact_cellot ===


Traceback (most recent call last):
  File "<string>", line 5, in <module>
  File "/n/home01/jzhou1125/.conda/envs/CellOT/lib/python3.9/site-packages/pandas/__init__.py", line 51, in <module>
    from pandas.core.api import (
  File "<frozen importlib._bootstrap>", line 1007, in _find_and_load
  File "<frozen importlib._bootstrap>", line 986, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 680, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 851, in exec_module
  File "<frozen importlib._bootstrap_external>", line 946, in get_code
  File "<frozen importlib._bootstrap_external>", line 1044, in get_data
KeyboardInterrupt


KeyboardInterrupt: 

## 2. Qualitative UMAP overlay per flavor

Stack all 5 cohorts in one fresh latent space (recompute per the 04-20 fresh-UMAP rule):
- atlas mouse, atlas human, BCG mouse, scGen-predicted, IMPACT-predicted

In [ ]:
def to_dense(X):
    return np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)


def umap_overlay(flavor, out_stem):
    atlas = ad.read_h5ad(DATA_DIR / f"hvg_{flavor}_atlas_full_v07.h5ad")
    bcg = ad.read_h5ad(DATA_DIR / f"bcg_mouse_aligned_{flavor}_v07.h5ad")
    pred_scgen = DATA_DIR / f"bcg_predicted_human_via_scgen_{flavor}.h5ad"
    pred_impact = DATA_DIR / f"bcg_predicted_human_via_impact_cellot_{flavor}.h5ad"

    blocks = []; labels = []
    am = atlas[atlas.obs["condition"] == "mouse"]
    ah = atlas[atlas.obs["condition"] == "human"]
    if len(am) > 0:
        blocks.append(to_dense(am.X)); labels += ["atlas mouse"] * len(am)
    if len(ah) > 0:
        blocks.append(to_dense(ah.X)); labels += ["atlas human"] * len(ah)
    blocks.append(to_dense(bcg.X)); labels += ["BCG mouse"] * len(bcg)
    if pred_scgen.exists():
        s = ad.read_h5ad(pred_scgen)
        blocks.append(to_dense(s.X)); labels += ["scGen pred"] * len(s)
    if pred_impact.exists():
        i = ad.read_h5ad(pred_impact)
        blocks.append(to_dense(i.X)); labels += ["IMPACT pred"] * len(i)

    X = np.vstack(blocks)
    obs = pd.DataFrame({"label": labels})
    a = ad.AnnData(X=X, obs=obs)

    sc.pp.pca(a, n_comps=min(50, a.n_vars - 1, a.n_obs - 1))
    sc.pp.neighbors(a, n_neighbors=min(15, a.n_obs - 1))
    sc.tl.umap(a)

    fig, ax = plt.subplots(figsize=(8, 7))
    palette = {"atlas mouse": "#bbbbbb", "atlas human": "tab:green",
               "BCG mouse": "tab:blue", "scGen pred": "tab:orange",
               "IMPACT pred": "tab:red"}
    for lab, color in palette.items():
        m = (a.obs["label"] == lab).values
        if not m.sum():
            continue
        ax.scatter(a.obsm["X_umap"][m, 0], a.obsm["X_umap"][m, 1],
                   s=4, alpha=0.4, c=color, label=f"{lab} (n={m.sum()})", edgecolors="none")
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
    ax.set_title(f"BCG prediction overlay — {flavor}\n(fresh PCA+UMAP in this gene space)", fontsize=11)
    ax.legend(fontsize=9, markerscale=3, loc="best")
    fig.tight_layout()
    for ext in ("pdf", "png"):
        out = FIG_DIR / f"{out_stem}.{ext}"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"  saved {out}")
    plt.close(fig)


for flavor in FLAVORS:
    print(f"\n=== UMAP overlay for {flavor} ===")
    umap_overlay(flavor, f"umap_overlay_{flavor}")

print("\nDone.")


=== UMAP overlay for seurat_v3 ===


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/bcg_prediction_outputs/figures/umap_overlay_seurat_v3.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/bcg_prediction_outputs/figures/umap_overlay_seurat_v3.png

=== UMAP overlay for pearson_residuals ===


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/bcg_prediction_outputs/figures/umap_overlay_pearson_residuals.pdf


  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/bcg_prediction_outputs/figures/umap_overlay_pearson_residuals.png

Done.
